# RMIT Hackathon 2026 — Challenge 1 | XLM-R large, chỉ dữ liệu BTC

**Model được chọn (whitelist BTC):** `FacebookAI/xlm-roberta-large`
(bản Kaggle: `aisuko/xlm-roberta-large`). Chỉ train bằng `train.csv` của BTC; không dữ liệu ngoài,
không dữ liệu tổng hợp, `test.csv` chỉ dùng để dự đoán.

**Cách train:** fine-tune toàn bộ 24 layer encoder + head mới, **đóng băng embedding**
(256M tham số từ vựng chiếm gần nửa model; đóng băng để vừa 8 GB VRAM).
BF16 autocast + trọng số FP32, gradient checkpointing, batch 8 × accum 2 = 16, 512 token đầu+cuối,
layer-wise LR decay, weighted CE nhẹ, chọn checkpoint theo **macro-F1**.

Đã đo trên RTX 5060 Laptop 8 GB với cấu hình large ngẫu nhiên: peak ~6.6 GB, ~0.58 s/step ở batch dài nhất.
Chạy từng cell theo thứ tự (Shift+Enter). Notebook không tự nộp Kaggle.

## 0. Tải model về đúng chỗ

Đặt model vào `models/xlm-roberta-large/` trong thư mục dự án. Chỉ cần các file:
`config.json`, `model.safetensors`, `sentencepiece.bpe.model`, `tokenizer.json`.
Không cần tải `onnx/`, `tf_model.h5`, `flax_model.msgpack`, `pytorch_model.bin` (tốn vài GB vô ích).

```powershell
.\.venv\Scripts\hf.exe download FacebookAI/xlm-roberta-large config.json model.safetensors sentencepiece.bpe.model tokenizer.json --local-dir models\xlm-roberta-large
```

Nếu tải bản Kaggle `aisuko/xlm-roberta-large`, giải nén sao cho `config.json` nằm trong
`models/xlm-roberta-large/` (hoặc thư mục con của nó, cell 3 tự tìm).
Nếu thư mục này chưa có, notebook sẽ tự tải từ Hugging Face vào cache.

In [ ]:
import sys, subprocess, platform
print('Python:', sys.version.split()[0], '|', sys.executable)
import torch, transformers
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| transformers:', transformers.__version__)
assert torch.cuda.is_available(), 'CUDA không hoạt động: chọn đúng kernel .venv (torch cu128).'
props = torch.cuda.get_device_properties(0)
print('GPU:', props.name, '| VRAM GB:', round(props.total_memory / 2**30, 2), '| BF16:', torch.cuda.is_bf16_supported())
free, total = torch.cuda.mem_get_info()
print('VRAM trống GB:', round(free / 2**30, 2), '(đóng app khác đang dùng GPU nếu < 7 GB)')

## 1. Cấu hình — đổi `RUN_NAME` mỗi khi đổi tham số

In [ ]:
import os, re, gc, json, time, random, datetime
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import (AutoConfig, AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback,
                          TrainerCallback, set_seed)

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
MODEL_ID = 'FacebookAI/xlm-roberta-large'
MODEL_DIR = ROOT / 'models' / 'xlm-roberta-large'
EXPECTED_LAYERS = 24                # xlm-roberta-large; chặn việc nạp nhầm bản base
RUN_NAME = 'xlmr_large_fold0_v1'
OUT = ROOT / 'runs' / RUN_NAME

SEED = 42
N_SPLITS, FOLD = 5, 0               # cùng split với lần chạy Colab (StratifiedKFold language+label, seed 42)
MAX_LENGTH = 512
HEAD_FRACTION = 0.67                # prompt dài: giữ 67% token đầu + 33% token cuối
BATCH_SIZE, GRAD_ACCUM = 8, 2       # effective batch 16
EVAL_BATCH_SIZE = 32
EPOCHS = 4
LEARNING_RATE = 1.5e-5              # LR của layer trên cùng
LAYERWISE_LR_DECAY = 0.92           # layer thấp học chậm hơn
HEAD_LR_MULTIPLIER = 5.0            # head mới khởi tạo học nhanh hơn
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
FREEZE_EMBEDDINGS = True
FREEZE_LAYERS = 0                   # nếu OOM: đặt 6 (đóng băng 6 layer dưới cùng)
CLASS_WEIGHT_POWER = 0.3            # 0 = CE thường; 0.3 ~ trọng số [1.0, 1.23, 1.92]
EARLY_STOP_PATIENCE = 2
BIAS_MODE = 'auto'                  # 'auto' | 'off': chỉnh bias logits cho macro-F1 (cell 9)
REFIT_ON_FULL_TRAIN = False         # bật sau khi đã có kết quả fold 0 tốt (cell 10)

LABELS = ['BENIGN', 'JAILBREAK', 'PROMPT_INJECTION']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

assert (DATA_DIR / 'train.csv').exists(), 'Mở VS Code tại thư mục dự án (chứa data/train.csv).'
OUT.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
USE_BF16 = torch.cuda.is_bf16_supported()
CONFIG = {k: (str(v) if isinstance(v, Path) else v) for k, v in globals().items()
          if k.isupper() and isinstance(v, (int, float, str, bool, list, Path))}
(OUT / 'config.json').write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')
print('Run:', OUT)

## 2. Đọc dữ liệu BTC và tạo split validation

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv', dtype={'id': str}, keep_default_na=False)
test_df = pd.read_csv(DATA_DIR / 'test.csv', dtype={'id': str}, keep_default_na=False)
sample_df = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'id': str}, keep_default_na=False)

assert list(train_df.columns) == ['id', 'text', 'language', 'label']
assert list(test_df.columns) == ['id', 'text', 'language']
assert len(train_df) == 14000 and len(test_df) == len(sample_df) == 6000
assert train_df.id.is_unique and test_df.id.is_unique and not set(train_df.id) & set(test_df.id)
assert set(sample_df.id) == set(test_df.id)
for df in (train_df, test_df):
    df['language'] = df.language.str.strip().str.lower()
train_df['label'] = train_df.label.str.strip().str.upper()
assert set(train_df.label) == set(LABELS)
train_labels = train_df.label.map(LABEL2ID).to_numpy(np.int64)

strata = train_df.language + '__' + train_df.label
folds = list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(train_df, strata))
tri, vai = folds[FOLD]
assert not set(tri) & set(vai)
print(f'Train {len(tri):,} | Validation {len(vai):,}')
display(pd.crosstab(train_df.iloc[vai].language, train_df.iloc[vai].label, margins=True))

## 3. Model và tokenizer

In [ ]:
def find_local_model(folder):
    folder = Path(folder)
    if not folder.exists():
        return None
    for cfg in sorted(folder.rglob('config.json'), key=lambda p: len(p.parts)):
        d = cfg.parent
        has_weights = any((d / f).exists() for f in
                          ['model.safetensors', 'model.safetensors.index.json', 'pytorch_model.bin'])
        has_tokenizer = any((d / f).exists() for f in ['sentencepiece.bpe.model', 'tokenizer.json'])
        if has_weights and has_tokenizer:
            return d
    return None

local_dir = find_local_model(MODEL_DIR)
MODEL_SOURCE = str(local_dir) if local_dir else MODEL_ID
print('Nạp model từ:', MODEL_SOURCE, '' if local_dir else '(chưa thấy bản local, tải từ Hugging Face)')
model_config = AutoConfig.from_pretrained(MODEL_SOURCE)
assert model_config.model_type == 'xlm-roberta', model_config.model_type
assert model_config.num_hidden_layers == EXPECTED_LAYERS, f'Đây không phải xlm-roberta-large ({model_config.num_hidden_layers} layer).'
tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE)
print(type(tokenizer).__name__, '| hidden:', model_config.hidden_size, '| layers:', model_config.num_hidden_layers)

## 4. Token hóa một lần — giữ cả đầu và cuối prompt dài

In [ ]:
MAX_PAYLOAD = MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False)

def encode_prompt(text):
    ids = tokenizer(text, add_special_tokens=False, verbose=False)['input_ids']
    n = len(ids)
    if n > MAX_PAYLOAD:
        head = round(MAX_PAYLOAD * HEAD_FRACTION)
        ids = ids[:head] + ids[-(MAX_PAYLOAD - head):]
    ids = tokenizer.build_inputs_with_special_tokens(ids)
    return {'input_ids': ids, 'attention_mask': [1] * len(ids)}, n

t0 = time.perf_counter()
train_enc, train_len = zip(*[encode_prompt(t) for t in train_df.text])
test_enc, test_len = zip(*[encode_prompt(t) for t in test_df.text])
train_len, test_len = np.array(train_len), np.array(test_len)
print(f'Token hóa {len(train_enc) + len(test_enc):,} prompt trong {time.perf_counter() - t0:.1f}s')
print('Số token p50/p90/p99/max:', np.quantile(train_len, [.5, .9, .99, 1]).astype(int))
print(f'Bị cắt: train {(train_len > MAX_PAYLOAD).mean():.1%} | test {(test_len > MAX_PAYLOAD).mean():.1%}')


class PromptDataset(Dataset):
    def __init__(self, encodings, indices, labels=None):
        self.encodings, self.indices, self.labels = encodings, [int(i) for i in indices], labels
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        item = dict(self.encodings[j])
        if self.labels is not None:
            item['labels'] = int(self.labels[j])
        return item

collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
test_ds = PromptDataset(test_enc, range(len(test_enc)))

## 5. Thành phần huấn luyện

**Đọc bảng log cho đúng:** cột *Training Loss* trong bảng của Trainer **không phải** trung bình cả epoch,
mà là loss trung bình của `logging_steps` bước gần nhất trước khi eval. Loss giảm mạnh epoch 1→2 rồi chậm lại
là đường cong bình thường (model học phần dễ trước; lớp hiếm PROMPT_INJECTION "bật lên" muộn hơn).
Notebook này in thêm `train_loss_avg_epoch` = trung bình loss thật của cả epoch để so với `val_loss`.
Label smoothing đã bỏ (nó đặt sàn loss ~0.08 và làm khó so sánh).

Nếu macro-F1 sau epoch 1 < 0.40 (gần mức đoán toàn BENIGN = 0.26), XLM-R large đã bị *collapse*:
training tự dừng; đổi `SEED` hoặc giảm `LEARNING_RATE` còn 1e-5 với `RUN_NAME` mới.

In [ ]:
def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_SOURCE, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID, dtype=torch.float32,
        ignore_mismatched_sizes=True)       # luôn tạo head 3 lớp mới
    if FREEZE_EMBEDDINGS:
        for p in model.roberta.embeddings.parameters():
            p.requires_grad = False
    for layer in model.roberta.encoder.layer[:FREEZE_LAYERS]:
        for p in layer.parameters():
            p.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Tham số train: {trainable / 1e6:.1f}M / {total / 1e6:.1f}M')
    return model


def class_weights_for(indices):
    if not CLASS_WEIGHT_POWER:
        return None
    freq = np.bincount(train_labels[indices], minlength=3).astype(float)
    w = (freq.max() / freq) ** CLASS_WEIGHT_POWER
    w = w / w.mean()
    print('Class weights:', dict(zip(LABELS, w.round(3))))
    return torch.tensor(w, dtype=torch.float32)


class PromptTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.model_accepts_loss_kwargs = False   # loss tự tính trung bình trên từng micro-batch

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        weight = None if self.class_weights is None else self.class_weights.to(outputs.logits.device)
        loss = F.cross_entropy(outputs.logits.float(), labels, weight=weight)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        if self.optimizer is None:
            n_layers = self.model.config.num_hidden_layers
            groups = {}
            for name, p in self.model.named_parameters():
                if not p.requires_grad:
                    continue
                m = re.search(r'encoder\.layer\.(\d+)\.', name)
                if m:
                    lr = LEARNING_RATE * LAYERWISE_LR_DECAY ** (n_layers - 1 - int(m.group(1)))
                elif name.startswith('classifier'):
                    lr = LEARNING_RATE * HEAD_LR_MULTIPLIER
                else:
                    lr = LEARNING_RATE * LAYERWISE_LR_DECAY ** n_layers
                no_decay = name.endswith('bias') or 'LayerNorm' in name
                groups.setdefault((lr, 0.0 if no_decay else WEIGHT_DECAY), []).append(p)
            self.optimizer = torch.optim.AdamW(
                [{'params': ps, 'lr': lr, 'weight_decay': wd} for (lr, wd), ps in groups.items()],
                lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-6)
            lrs = [lr for lr, _ in groups]
            print(f'Optimizer: {len(groups)} nhóm, LR {min(lrs):.2e} → {max(lrs):.2e}')
        return self.optimizer


def compute_metrics(eval_pred):
    logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    f1 = f1_score(eval_pred.label_ids, logits.argmax(-1), labels=[0, 1, 2], average=None, zero_division=0)
    return {'macro_f1': float(f1.mean()), 'f1_benign': float(f1[0]),
            'f1_jailbreak': float(f1[1]), 'f1_injection': float(f1[2])}


class EpochSummary(TrainerCallback):
    """In loss trung bình cả epoch và dừng sớm nếu model collapse."""
    def __init__(self):
        self.losses = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self.losses.append(logs['loss'])
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        avg = float(np.mean(self.losses)) if self.losses else float('nan')
        self.losses = []
        print(f"[epoch {state.epoch:.2f}] train_loss_avg_epoch={avg:.4f} | val_loss={metrics['eval_loss']:.4f} | "
              f"macro_f1={metrics['eval_macro_f1']:.4f} (B {metrics['eval_f1_benign']:.3f} / "
              f"J {metrics['eval_f1_jailbreak']:.3f} / PI {metrics['eval_f1_injection']:.3f})")
        if state.epoch <= 1.01 and metrics['eval_macro_f1'] < 0.40:
            print('CẢNH BÁO: model collapse. Dừng train; đổi SEED hoặc LEARNING_RATE=1e-5 với RUN_NAME mới.')
            control.should_training_stop = True


def make_args(output_dir, do_eval=True, epochs=EPOCHS):
    return TrainingArguments(
        output_dir=str(output_dir), num_train_epochs=epochs, learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM, weight_decay=WEIGHT_DECAY,
        lr_scheduler_type='cosine', warmup_ratio=WARMUP_RATIO, max_grad_norm=1.0,
        bf16=USE_BF16, fp16=not USE_BF16,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
        eval_strategy='epoch' if do_eval else 'no', save_strategy='epoch' if do_eval else 'no',
        save_only_model=True, save_total_limit=1, load_best_model_at_end=do_eval,
        metric_for_best_model='eval_macro_f1' if do_eval else None, greater_is_better=True,
        logging_strategy='steps', logging_steps=50,
        group_by_length=True,               # gom prompt dài gần nhau, giảm padding
        dataloader_num_workers=0,           # Windows + notebook: worker process không pickle được class trong notebook
        report_to='none', seed=SEED, data_seed=SEED, remove_unused_columns=False)


def logits_of(output):
    p = output.predictions
    return np.asarray(p[0] if isinstance(p, tuple) else p, dtype=np.float32)

## 6. Smoke test — đo VRAM và tốc độ với batch dài nhất (~1 phút)

Nếu báo OOM: restart kernel, đặt `FREEZE_LAYERS = 6` (hoặc `MAX_LENGTH = 384`) và đổi `RUN_NAME`.

In [ ]:
model = build_model().cuda()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.train()
longest = np.argsort(train_len)[-BATCH_SIZE:]
batch = collator([{**train_enc[i], 'labels': int(train_labels[i])} for i in longest])
batch = {k: v.cuda() for k, v in batch.items()}
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-7)
torch.cuda.reset_peak_memory_stats()
times = []
for _ in range(4):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.autocast('cuda', dtype=torch.bfloat16 if USE_BF16 else torch.float16):
        logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
    loss = F.cross_entropy(logits.float(), batch['labels'])
    loss.backward(); optimizer.step(); optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize(); times.append(time.perf_counter() - t0)
peak = torch.cuda.max_memory_allocated() / 2**30
step = float(np.median(times[1:]))
print(f'Shape batch: {tuple(batch["input_ids"].shape)} | loss {loss.item():.3f} | peak VRAM {peak:.2f} GB')
print(f'{step:.2f} s/batch dài nhất → trần thời gian 1 epoch ≈ {len(tri) / BATCH_SIZE * step / 60:.0f} phút '
      '(thực tế nhanh hơn nhiều vì đa số prompt ngắn)')
del model, optimizer, batch, logits, loss
gc.collect(); torch.cuda.empty_cache()

## 7. Train fold 0 — chọn checkpoint theo macro-F1 validation

In [ ]:
model = build_model()
train_ds = PromptDataset(train_enc, tri, train_labels)
val_ds = PromptDataset(train_enc, vai, train_labels)
trainer = PromptTrainer(
    model=model, args=make_args(OUT / 'checkpoints'), train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics,
    class_weights=class_weights_for(tri),
    callbacks=[EarlyStoppingCallback(EARLY_STOP_PATIENCE), EpochSummary()])
t0 = time.perf_counter()
trainer.train()
train_minutes = (time.perf_counter() - t0) / 60

history = [(round(e['epoch']), e['eval_macro_f1']) for e in trainer.state.log_history if 'eval_macro_f1' in e]
best_epoch, best_f1 = max(history, key=lambda x: x[1])
print(f'Xong sau {train_minutes:.1f} phút | best epoch {best_epoch} | val macro-F1 {best_f1:.4f}')

val_logits = logits_of(trainer.predict(val_ds))
test_logits = logits_of(trainer.predict(test_ds))
np.save(OUT / 'val_logits.npy', val_logits)
np.save(OUT / 'test_logits_fold0.npy', test_logits)
trainer.save_model(str(OUT / 'best_model'))
tokenizer.save_pretrained(str(OUT / 'best_model'))
pd.DataFrame(trainer.state.log_history).to_csv(OUT / 'train_log.csv', index=False)
(OUT / 'fold0_summary.json').write_text(json.dumps({
    'best_epoch': best_epoch, 'val_macro_f1': best_f1, 'history': history,
    'train_minutes': train_minutes, 'peak_vram_gb': torch.cuda.max_memory_allocated() / 2**30}, indent=2))

del trainer, model
gc.collect(); torch.cuda.empty_cache()

## 8. Phân tích lỗi trên validation

In [ ]:
y_val = train_labels[vai]
pred_val = val_logits.argmax(1)
print(classification_report(y_val, pred_val, labels=[0, 1, 2], target_names=LABELS, digits=4, zero_division=0))
print('Confusion matrix (hàng = thật, cột = dự đoán):')
display(pd.DataFrame(confusion_matrix(y_val, pred_val, labels=[0, 1, 2]), index=LABELS, columns=LABELS))

val_frame = train_df.iloc[vai][['id', 'language', 'label', 'text']].copy()
probs = np.exp(val_logits - val_logits.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
val_frame['prediction'] = [LABELS[i] for i in pred_val]
val_frame['confidence'] = probs.max(1)
rows = []
for lang, g in val_frame.groupby('language'):
    f1 = f1_score(g.label, g.prediction, labels=LABELS, average=None, zero_division=0)
    rows.append({'language': lang, 'n': len(g), 'macro_f1': f1.mean(),
                 **{'f1_' + l.lower(): v for l, v in zip(LABELS, f1)}})
display(pd.DataFrame(rows).round(4))
errors = val_frame[val_frame.label != val_frame.prediction].sort_values('confidence', ascending=False)
errors.to_csv(OUT / 'validation_errors.csv', index=False)
print(f'{len(errors)} lỗi, lưu tại', OUT / 'validation_errors.csv')
display(errors[['language', 'label', 'prediction', 'confidence', 'text']].head(15))

## 9. Chỉnh bias logits cho macro-F1 (có kiểm chứng chéo)

Cộng một hằng số vào logit JAILBREAK / PROMPT_INJECTION để cân bằng precision–recall từng lớp.
Để tránh overfit validation: chia validation làm 2 nửa, tìm bias trên nửa này rồi chấm trên nửa kia.
Chỉ áp dụng khi lợi ích kiểm chứng chéo > 0.

In [ ]:
GRID = np.round(np.arange(-1.5, 1.501, 0.1), 2)

def macro(y, logits, bias=(0.0, 0.0, 0.0)):
    return f1_score(y, (logits + np.asarray(bias)).argmax(1), labels=[0, 1, 2], average='macro', zero_division=0)

def search_bias(y, logits):
    best = (macro(y, logits), (0.0, 0.0, 0.0))
    for a in GRID:
        for b in GRID:
            s = macro(y, logits, (0.0, a, b))
            if s > best[0] + 1e-9:
                best = (s, (0.0, float(a), float(b)))
    return best

chosen_bias = np.zeros(3, dtype=np.float32)
if BIAS_MODE == 'auto':
    halves = list(StratifiedKFold(2, shuffle=True, random_state=SEED).split(vai, strata.iloc[vai]))
    gains = []
    for fit_idx, check_idx in halves:
        _, bias = search_bias(y_val[fit_idx], val_logits[fit_idx])
        gains.append(macro(y_val[check_idx], val_logits[check_idx], bias) - macro(y_val[check_idx], val_logits[check_idx]))
    full_score, full_bias = search_bias(y_val, val_logits)
    print(f'Macro-F1 không bias: {macro(y_val, val_logits):.4f} | bias {full_bias} trên toàn validation: {full_score:.4f} (lạc quan)')
    print('Lợi ích kiểm chứng chéo từng nửa:', np.round(gains, 4))
    if np.mean(gains) > 0:
        chosen_bias = np.asarray(full_bias, dtype=np.float32)
        print('→ Áp dụng bias', chosen_bias)
    else:
        print('→ Không áp dụng bias (không có lợi ích ổn định)')

## 10. (Tùy chọn) Train lại trên toàn bộ 14.000 dòng

Chỉ bật `REFIT_ON_FULL_TRAIN = True` (cell 1) sau khi fold 0 cho kết quả tốt. Dùng đúng số epoch tốt nhất
của fold 0, không có validation nên không chọn checkpoint. Thêm 20% dữ liệu thường giúp ít nhưng ổn định;
tốn thêm khoảng bằng thời gian train fold 0.

In [ ]:
if REFIT_ON_FULL_TRAIN:
    set_seed(SEED)
    all_idx = np.arange(len(train_df))
    model = build_model()
    trainer = PromptTrainer(
        model=model, args=make_args(OUT / 'refit', do_eval=False, epochs=best_epoch),
        train_dataset=PromptDataset(train_enc, all_idx, train_labels),
        processing_class=tokenizer, data_collator=collator, class_weights=class_weights_for(all_idx))
    trainer.train()
    test_logits = logits_of(trainer.predict(test_ds))
    np.save(OUT / 'test_logits_refit.npy', test_logits)
    trainer.save_model(str(OUT / 'refit_model'))
    tokenizer.save_pretrained(str(OUT / 'refit_model'))
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    print('Đã dùng model refit cho test.')
else:
    print('Bỏ qua refit; dùng model fold 0.')

## 11. Xuất submission

In [ ]:
pred = (test_logits + chosen_bias).argmax(1)
submission = pd.DataFrame({'id': test_df.id, 'label': [LABELS[i] for i in pred]})
submission = sample_df[['id']].merge(submission, on='id', how='left', validate='one_to_one')
assert len(submission) == 6000 and submission.id.is_unique and submission.label.isin(LABELS).all()
assert submission.id.tolist() == sample_df.id.tolist()

stamp = datetime.datetime.now().strftime('%m%d_%H%M')
path = OUT / f'submission_{stamp}.csv'
submission.to_csv(path, index=False)
check = pd.read_csv(path, dtype={'id': str}, keep_default_na=False)
assert check.shape == (6000, 2) and list(check.columns) == ['id', 'label']
print('Đã lưu:', path)
print('Bias áp dụng:', chosen_bias, '| refit:', REFIT_ON_FULL_TRAIN)
display(pd.DataFrame({'test_pred': submission.label.value_counts(normalize=True).round(3),
                      'train_prior': train_df.label.value_counts(normalize=True).round(3)}))